In [ ]:
!git clone https://github.com/Mohamedmagdy21/sentiment_analyzer.git

%cd sentiment_analyzer

# Patch: remove tokenizer kwarg from Trainer() (transformers 5.x compat)
with open("training/trainers/hf_trainer.py", "r") as f:
    content = f.read()
content = content.replace(
    "            tokenizer=self.tokenizer\n",
    ""
)
with open("training/trainers/hf_trainer.py", "w") as f:
    f.write(content)
print("Patched hf_trainer.py for transformers 5.x")

# Add max_steps=100 for CPU training (Kaggle GPU quota exhausted)
with open("training/trainers/hf_trainer.py", "r") as f:
    content = f.read()
content = content.replace(
    "            num_train_epochs=1,\n",
    "            num_train_epochs=1,\n            max_steps=100,\n"
)
with open("training/trainers/hf_trainer.py", "w") as f:
    f.write(content)
print("Set max_steps=100 for CPU training")

# Fix: import hf_trainer in __init__.py so Hydra can resolve it
with open("training/trainers/__init__.py", "w") as f:
    f.write("from training.trainers import hf_trainer\n")
print("Fixed trainers/__init__.py")

!pip install -r requirements.txt
# Remove torchvision (transformers imports it but we don't need it — causes nms operator crash)
!pip uninstall torchvision -y

In [ ]:
import os
os.makedirs("Data/raw/sent", exist_ok=True)
!cp /kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv \
    Data/raw/sent/training.1600000.processed.noemoticon.csv
print("Copied raw Twitter data from Kaggle input.")

In [ ]:
!python -m preprocessing.preprocess dataset=twitter hydra.run.dir=.

In [ ]:
!CUDA_VISIBLE_DEVICES="" HYDRA_FULL_ERROR=1 python -m training.train \
    dataset=twitter \
    model=twitter_roberta \
    hydra.run.dir=.